In [1]:
from llama_index.core import SimpleDirectoryReader

input_dirs = {
    "roo-coder": "../TaskClient/docs/roo-docs", 
    "cline": "../TaskClient/docs/cline-logs"
}
documents = []

for agent_name, path in input_dirs.items():
    # פונקציית עזר שמוסיפה מטא-דאטה לכל קובץ שנקרא מהתיקייה
    def add_agent_metadata(file_path):
        return {
            "agent_tool": agent_name,
            "source_folder": path
        }

    reader = SimpleDirectoryReader(
        input_dir=path, 
        recursive=True, 
        required_exts=[".md"],
        file_metadata=add_agent_metadata # כאן קורה הקסם
    )
    documents.extend(reader.load_data())

print(f"נטענו {len(documents)} מסמכים עם תיוג Agent.")



from llama_index.core.node_parser import MarkdownNodeParser

# יצירת הפארסר שמזהה כותרות (H1, H2, וכו')
parser = MarkdownNodeParser()

# פירוק המסמכים לצמתים מבוססי מבנה
nodes = parser.get_nodes_from_documents(documents)

# טיפ: כדאי לבדוק את ה-metadata של הצמתים שנוצרו
# כל צומת יכיל כעת מידע על ה-Header שהוא שייך אליו
print(nodes[0].metadata)



from llama_index.core import VectorStoreIndex, Settings
from llama_index.embeddings.cohere import CohereEmbedding
from llama_index.core.node_parser import MarkdownNodeParser
import os
from dotenv import load_dotenv
load_dotenv()
api_key = os.getenv("COHERE_API_KEY")
# 1. הגדרת מודל ה-Embedding של Cohere
# המודל 'embed-multilingual-v3.0' מעולה למסמכים טכניים בעברית ובאנגלית
embed_model = CohereEmbedding(
    cohere_api_key=api_key,
    model_name="embed-multilingual-v3.0",
)

# 2. עדכון הגדרות הגלובליות של LlamaIndex
Settings.embed_model = embed_model
# כאן כדאי להגדיר גם את ה-chunk_size אם את לא משתמשת ב-Parser חיצוני
Settings.chunk_size = 512 

# 3. פירוק לצמתים (Nodes) באמצעות MarkdownNodeParser כפי שסיכמנו
parser = MarkdownNodeParser()
nodes = parser.get_nodes_from_documents(documents)

# 4. יצירת האינדקס הוקטורי
# בשלב זה המערכת שולחת את הטקסט ל-Cohere ומקבלת וקטורים בחזרה
index = VectorStoreIndex(nodes)

print(f"הסתיים אינדוקס של {len(nodes)} צמתים.")
















נטענו 9 מסמכים עם תיוג Agent.
{'agent_tool': 'roo-coder', 'source_folder': '../TaskClient/docs/roo-docs', 'header_path': '/'}


2026-03-18 21:43:24,078 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"
2026-03-18 21:43:30,978 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"
2026-03-18 21:43:33,316 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"
2026-03-18 21:43:35,311 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"
2026-03-18 21:43:37,175 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"
2026-03-18 21:43:40,605 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"
2026-03-18 21:43:42,504 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"
2026-03-18 21:43:43,206 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"
2026-03-18 21:43:43,986 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"
2026-03-18 21:43:44,628 - INFO - HTTP Request: POST https://api.cohere.com/v2/embe

הסתיים אינדוקס של 119 צמתים.


In [2]:
import os
from pinecone import Pinecone
from llama_index.vector_stores.pinecone import PineconeVectorStore
from llama_index.core import StorageContext
from dotenv import load_dotenv
load_dotenv()
# 1. הגדר את המפתח כאן (החלף את המחרוזת pcsk_... במפתח האמיתי שלך)
MY_API_KEY = os.getenv("PINECONE_API_KEY")

# 2. התחברות ל-Pinecone
pc = Pinecone(
    api_key=MY_API_KEY, 
    ssl_verify=False
)

# 3. הגדרת ה-Index Host והאינדקס
INDEX_HOST = "https://rag-index-2mwyo1g.svc.aped-4627-b74a.pinecone.io"
pinecone_index = pc.Index(name="rag-index", host=INDEX_HOST)

# 4. הגדרת ה-Vector Store - שים לב להעברת המפתח גם כאן!
vector_store = PineconeVectorStore(
    pinecone_index=pinecone_index,
    api_key=MY_API_KEY  # חשוב להעביר את המפתח גם ל-LlamaIndex
)

storage_context = StorageContext.from_defaults(vector_store=vector_store)

# 5. יצירת האינדקס
index = VectorStoreIndex(
    nodes, 
    storage_context=storage_context,
    show_progress=True
)

C:\Users\user1\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Upserted vectors:   0%|          | 0/119 [00:00<?, ?it/s]C:\Users\user1\AppData\Roaming\Python\Python312\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'rag-index-2mwyo1g.svc.aped-4627-b74a.pinecone.io'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
Upserted vectors:  84%|████████▍ | 100/119 [00:17<00:03,  5.81it/s]C:\Users\user1\AppData\Roaming\Python\Python312\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'rag-index-2mwyo1g.svc.aped-4627-b74a.pinecone.io'. Adding certificate verific

In [17]:
import os
from pinecone import Pinecone
from llama_index.vector_stores.pinecone import PineconeVectorStore
from llama_index.core import StorageContext, VectorStoreIndex
from dotenv import load_dotenv
load_dotenv()
# --- שלב הבדיקה הקריטי ---
# 1. העתיקי לכאן את המפתח שמתחיל ב-pcsk_ (זה של Pinecone!)
PINECONE_KEY = "pcsk_4vR48K_R2joMdsNbZR7CjUhA9k2sgcptgoyYqvuVmyUqF5JN6YMvEf8K2VviTPDMekoHxr"
#PINECONE_KEY=os.getenv("PINECONE_API_KEY")
# 2. ודאי שה-HOST נכון (העתיקי מה-Console של Pinecone אם יש ספק)
INDEX_HOST = "https://rag-index-2mwyo1g.svc.aped-4627-b74a.pinecone.io"

# --- חיבור ישיר ופשוט ---
try:
    # אתחול ה-Client
    pc = Pinecone(api_key=PINECONE_KEY, ssl_verify=False)
    
    # חיבור לאינדקס הספציפי
    pinecone_index = pc.Index(name="rag-index", host=INDEX_HOST)
    
    # בדיקת חיבור בסיסית - ננסה לבקש מהאינדקס את התיאור שלו
    stats = pinecone_index.describe_index_stats()
    print("✅ חיבור ל-Pinecone הצליח! סטטיסטיקות אינדקס:", stats)

    # הגדרת ה-Vector Store
    vector_store = PineconeVectorStore(
        pinecone_index=pinecone_index,
        api_key=PINECONE_KEY
    )

    # יצירת ה-Context
    storage_context = StorageContext.from_defaults(vector_store=vector_store)

    # העלאת הצמתים (Nodes)
    index = VectorStoreIndex(
        nodes, 
        storage_context=storage_context,
        show_progress=True
    )
    print("🚀 סיימנו! הנתונים עלו וה-index מוכן לעבודה.")

except Exception as e:
    print(f"❌ עדיין יש בעיה. הודעת השגיאה המלאה: {e}")

C:\Users\user1\AppData\Roaming\Python\Python312\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'rag-index-2mwyo1g.svc.aped-4627-b74a.pinecone.io'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


✅ חיבור ל-Pinecone הצליח! סטטיסטיקות אינדקס: {'dimension': 1024,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 476}},
 'total_vector_count': 476,
 'vector_type': 'dense'}


Upserted vectors:   0%|          | 0/119 [00:00<?, ?it/s]C:\Users\user1\AppData\Roaming\Python\Python312\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'rag-index-2mwyo1g.svc.aped-4627-b74a.pinecone.io'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
Upserted vectors:  84%|████████▍ | 100/119 [00:35<00:06,  2.79it/s]C:\Users\user1\AppData\Roaming\Python\Python312\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'rag-index-2mwyo1g.svc.aped-4627-b74a.pinecone.io'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
Upserted vectors: 100%|██████████| 119/119 [00:42<00:00,  2.78it/s]

🚀 סיימנו! הנתונים עלו וה-index מוכן לעבודה.


In [3]:
from llama_index.core import get_response_synthesizer
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.postprocessor import SimilarityPostprocessor
from llama_index.llms.cohere import Cohere
from llama_index.core.postprocessor import SimilarityPostprocessor
# 1. הגדרת ה-LLM (לניסוח התשובה הסופית)
from llama_index.core import Settings

from llama_index.llms.cohere import Cohere
from llama_index.core import Settings

# האופציה המומלצת - מודל command-r בגרסה מעודכנת
llm = Cohere(
    api_key=os.getenv("COHERE_API_KEY"), 
    model="command-r-08-2024" # או פשוט "command-r-v2" בהתאם לעדכוני Cohere האחרונים
)

# הגדרה גלובלית
Settings.llm = llm

# חשוב: אם כבר יצרת את ה-query_engine, צריך ליצור אותו מחדש 
# כדי שהוא "יידע" להשתמש ב-llm החדש:
query_engine = index.as_query_engine(
    similarity_top_k=5,
    llm=llm # הזרקה ישירה של המודל המעודכן
)

# 2. הגדרת ה-Retriever: כמה צמתים לשלוף מ-Pinecone?
retriever = VectorIndexRetriever(
    index=index,
    similarity_top_k=5, # שולף את 5 ה-chunks הכי רלוונטיים
)

# 3. הגדרת ה-Synthesizer: איך "להלחים" את המידע לתשובה
response_synthesizer = get_response_synthesizer(
    llm=llm,
    response_mode="compact" # משלב את ה-chunks בצורה חסכונית ב-tokens
)


# 1. נוריד את הסף ל-0.45 כדי לאפשר לצמתים שמצאנו לעבור
# או שפשוט נסיר את ה-Postprocessor זמנית לבדיקה
node_postprocessors = [
    SimilarityPostprocessor(similarity_cutoff=0.45) 
]

# 2. עדכון ה-Query Engine עם הסף החדש
query_engine = RetrieverQueryEngine(
    retriever=retriever,
    response_synthesizer=response_synthesizer,
    node_postprocessors=node_postprocessors
)

In [7]:
import json
import os
from llama_index.core.program import LLMTextCompletionProgram
from pydantic import BaseModel, Field
from typing import List

# 1. הגדרת הסכמה לחילוץ (תואם ל-JSON שביקשת)
class Decision(BaseModel):
    id: str
    title: str
    summary: str
    tags: List[str]

class Rule(BaseModel):
    id: str
    rule: str
    scope: str

class ProjectData(BaseModel):
    decisions: List[Decision]
    rules: List[Rule]

# 2. פונקציה לסריקת תיקיית ה-MD וחילוץ המידע
def run_initial_extraction(directory_path):
    program = LLMTextCompletionProgram.from_defaults(
        output_cls=ProjectData,
        prompt_template_str="Analyze the following project documentation and extract decisions and rules in a structured format:\n{input_str}",
        llm=llm
    )
    
    final_data = {"decisions": [], "rules": []}
    
    for filename in os.listdir(directory_path):
        if filename.endswith(".md"):
            with open(os.path.join(directory_path, filename), 'r', encoding='utf-8') as f:
                content = f.read()
                # חילוץ מובנה מהקובץ
                extracted = program(input_str=content)
                final_data["decisions"].extend([d.dict() for d in extracted.decisions])
                final_data["rules"].extend([r.dict() for r in extracted.rules])
                
    return final_data

# 3. הרצה ועדכון ה-Workflow
# וודא שהנתיב מצביע לתיקיית המסמכים שלך (כמו .cursor/notes)
project_metadata = run_initial_extraction("../TaskClient/docs")

In [8]:
from pydantic import BaseModel, Field
from typing import List, Literal

class DecisionItem(BaseModel):
    title: str = Field(description="כותרת ההחלטה")
    summary: str = Field(description="תמצית ההחלטה")
    status: Literal["active", "deprecated", "pending"]

class RuleItem(BaseModel):
    rule: str = Field(description="הכלל או ההנחיה")
    scope: str = Field(description="תחום (UI, Backend, etc.)")

class StructuredData(BaseModel):
    decisions: List[DecisionItem]
    rules: List[RuleItem]

In [ ]:
from llama_index.core.program import LLMTextCompletionProgram
import json 
def extract_structured_data(documents):
    # הגדרת התוכנית לחילוץ
    program = LLMTextCompletionProgram.from_defaults(
        output_cls=StructuredData,
        prompt_template_str="""עבור הטקסט הבא ממסמכי הפרויקט, חלץ את כל ההחלטות והכללים המופיעים בו:
        ---
        {input_str}
        ---
        """,
        llm=llm # ה-LLM שהגדרת קודם
    )
    
    all_extracted_data = {"decisions": [], "rules": []}
    
    for doc in documents:
        result = program(input_str=doc.get_content())
        all_extracted_data["decisions"].extend(result.decisions)
        all_extracted_data["rules"].extend(result.rules)
        
    return all_extracted_data

# נניח ששמרנו את זה למשתנה גלובלי או לקובץ
project_metadata = extract_structured_data(documents)
# שמירה - המרת Pydantic objects לדיקשנריים
with open('structured_data.json', 'w', encoding='utf-8') as f:
    # המרת כל DecisionItem ו-RuleItem לדיקשנריים
    serializable_data = {
        "decisions": [d.model_dump() if hasattr(d, 'model_dump') else d.dict() for d in project_metadata["decisions"]],
        "rules": [r.model_dump() if hasattr(r, 'model_dump') else r.dict() for r in project_metadata["rules"]]
    }
    json.dump(serializable_data, f, ensure_ascii=False, indent=2)




2026-03-18 21:08:16,830 - INFO - HTTP Request: POST https://api.cohere.com/v1/chat "HTTP/1.1 200 OK"
2026-03-18 21:08:26,767 - INFO - HTTP Request: POST https://api.cohere.com/v1/chat "HTTP/1.1 200 OK"
2026-03-18 21:08:30,031 - INFO - HTTP Request: POST https://api.cohere.com/v1/chat "HTTP/1.1 200 OK"
2026-03-18 21:08:42,639 - INFO - HTTP Request: POST https://api.cohere.com/v1/chat "HTTP/1.1 200 OK"
2026-03-18 21:08:50,933 - INFO - HTTP Request: POST https://api.cohere.com/v1/chat "HTTP/1.1 200 OK"
2026-03-18 21:08:54,484 - INFO - HTTP Request: POST https://api.cohere.com/v1/chat "HTTP/1.1 200 OK"
2026-03-18 21:08:57,959 - INFO - HTTP Request: POST https://api.cohere.com/v1/chat "HTTP/1.1 200 OK"
2026-03-18 21:09:08,726 - INFO - HTTP Request: POST https://api.cohere.com/v1/chat "HTTP/1.1 200 OK"
2026-03-18 21:09:13,525 - INFO - HTTP Request: POST https://api.cohere.com/v1/chat "HTTP/1.1 200 OK"


In [ ]:
import json
import gradio as gr
from typing import List, Union
from llama_index.core.workflow import Workflow, step, StartEvent, StopEvent, Event
from llama_index.core import get_response_synthesizer

from llama_index.utils.workflow import draw_all_possible_flows
# --- 1. הגדרת האירועים (Events) ---
class StructuredRetrievalEvent(Event):
    query: str
    data: dict

class RetrievalEvent(Event):
    query: str

class ValidationEvent(Event):
    nodes: list
    query: str

class GenerationEvent(Event):
    nodes: list
    query: str

# --- 2. בניית ה-Workflow האג'נטי ---
class AgenticRAGWorkflow(Workflow):
    def __init__(self, retriever, synthesizer, structured_data, **kwargs):
        super().__init__(**kwargs)
        self.retriever = retriever
        self.synthesizer = synthesizer
        self.structured_data = structured_data

    @step
    async def ingest_and_route(self, ev: StartEvent) -> RetrievalEvent | StructuredRetrievalEvent | StopEvent:
        query = ev.get("query")
        
        # ולידציה בסיסית של הקלט
        if not query or len(query.strip()) < 3:
            return StopEvent(result="השאילתה קצרה מדי. אנא שאל שאלה מפורטת יותר.")
        
        # ניתוב חכם בעזרת LLM (מזהה כוונת משתמש לרשימות/זמן/כללים)
        router_prompt = f"""
        אתה מנתח כוונות עבור מערכת RAG. עליך להחליט לאן לנתב את השאלה:
        1. 'structured' - שאלות שמבקשות רשימות, כללים, הנחיות, החלטות, או שאלות מבוססות זמן.
        2. 'semantic' - שאלות ידע כלליות, הסברים, או חיפוש תוכן חופשי.
        
        השאלה: "{query}"
        ענה במילה אחת בלבד: structured או semantic.
        """
        
        # שימוש ב-llm הגלובלי שהגדרת
        decision_raw = llm.complete(router_prompt).text.strip().lower()
        decision = "structured" if "structured" in decision_raw else "semantic"
        
        print(f"--- Routing Decision: {decision} ---")

        if decision == "structured":
            return StructuredRetrievalEvent(query=query, data=self.structured_data)
        
        return RetrievalEvent(query=query)

    @step
    async def handle_structured(self, ev: StructuredRetrievalEvent) -> StopEvent:
        # שליפה מתוך ה-JSON שחולץ בשלב ה-Extraction
        prompt = f"""
        הנך עוזר טכני המסתמך על מאגר נתונים מובנה (JSON). 
        הנתונים: {ev.data}
        
        השאלה של המשתמש: "{ev.query}"
        
        הנחיות:
        1. ספק רשימות מלאות אם התבקשת.
        2. סנן לפי תאריכים אם השאלה כוללת תנאי זמן.
        3. אם המידע לא מופיע בנתונים המובנים, ציין זאת.
        """
        response = llm.complete(prompt) 
        return StopEvent(result=str(response))

    @step
    async def perform_retrieval(self, ev: RetrievalEvent) -> ValidationEvent:
        # חיפוש וקטורי רגיל (Pinecone/Vector DB)
        nodes = self.retriever.retrieve(ev.query)
        return ValidationEvent(nodes=nodes, query=ev.query)

    @step
    async def validate_results(self, ev: ValidationEvent) -> GenerationEvent | StopEvent:
        # בדיקת איכות התוצאות מהחיפוש הסמנטי
        if not ev.nodes or all(getattr(n, 'score', 0) < 0.45 for n in ev.nodes):
            return StopEvent(result="לא מצאתי מידע אמין מספיק במסמכי התיעוד כדי לענות על כך.")
        
        return GenerationEvent(nodes=ev.nodes, query=ev.query)

    @step
    async def generate_response(self, ev: GenerationEvent) -> StopEvent:
        # סינתזה של התשובה הסופית
        response = self.synthesizer.synthesize(query=ev.query, nodes=ev.nodes)
        return StopEvent(result=str(response))

# --- 3. איתחול הכלים והנתונים ---

# שליפת הנתונים המובנים מהקובץ (נוצר בשלב ה-Extraction)
try:
    with open('structured_data.json', 'r', encoding='utf-8') as f:
        project_metadata = json.load(f)
except FileNotFoundError:
    print("Warning: structured_data.json לא נמצא. מוודא שהרצת את ה-Extraction.")
    project_metadata = {"decisions": [], "rules": [], "warnings": []}

# הגדרת Retriever ו-Synthesizer (מבוסס על ה-index וה-llm הקיימים שלך)
retriever = index.as_retriever(similarity_top_k=5)
response_synthesizer = get_response_synthesizer(llm=llm)

# יצירת מופע ה-Workflow
rag_wf = AgenticRAGWorkflow(
    retriever=retriever, 
    synthesizer=response_synthesizer, 
    structured_data=project_metadata,
    timeout=60, 
    verbose=True
)

# --- 4. ממשק Gradio ---

async def workflow_chat(message, history):
    # הרצת ה-Workflow האסינכרוני
    result = await rag_wf.run(query=message)
    return str(result)

demo = gr.ChatInterface(
    fn=workflow_chat,
    title="🚀 Agentic Event-Driven RAG",
    description="מערכת RAG משולבת: חיפוש סמנטי + שליפת נתונים מובנים (JSON)"
)


# יצירת קובץ HTML שמציג את כל הצעדים והאירועים
draw_all_possible_flows(rag_wf, filename="workflow_schema.html")

# --- 5. הרצה ---
if __name__ == "__main__":
    demo.launch(share=True)

workflow_schema.html
* Running on local URL:  http://127.0.0.1:7860


2026-03-18 21:54:34,467 - INFO - HTTP Request: GET http://127.0.0.1:7860/gradio_api/startup-events "HTTP/1.1 200 OK"
2026-03-18 21:54:34,595 - INFO - HTTP Request: HEAD http://127.0.0.1:7860/ "HTTP/1.1 200 OK"
2026-03-18 21:54:40,493 - INFO - HTTP Request: GET https://api.gradio.app/v3/tunnel-request "HTTP/1.1 200 OK"
2026-03-18 21:54:42,658 - INFO - HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-initiated-analytics "HTTP/1.1 200 OK"
2026-03-18 21:54:45,707 - INFO - HTTP Request: GET https://cdn-media.huggingface.co/frpc-gradio-0.3/frpc_windows_amd64.exe "HTTP/1.1 200 OK"



Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


2026-03-18 21:57:32,564 - INFO - HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-error-analytics "HTTP/1.1 200 OK"
2026-03-18 21:57:32,591 - INFO - HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-launched-telemetry "HTTP/1.1 200 OK"
